# Stellar Classification: Deep-Dive Exploratory Data Analysis (EDA)

This notebook executes a rigorous analysis of the photometric and positional attributes of stars, galaxies, and quasars from Sloan Digital Sky Survey (SDSS)-like data. Our objective is to identify key physical properties (redshift separation, color index distributions, celestial coordinates) and transform them into predictive features for gradient boosted trees.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plot styling to match project standards (Viridis default)
sns.set_theme(style="whitegrid")
sns.set_palette("viridis")

## 1. Data Dimensions & Multi-Dataset Structure

We begin by loading and loading the metadata of `train.csv`, `test.csv`, and `sample_submission.csv`. The data contains over 577,000 stellar observations.
We utilize dynamic environment detection to handle both Kaggle environment routes (`/kaggle/input/competitions/playground-series-s6e6`) and local developer environments (`../data`) seamlessly without manual code modifications.

In [ ]:
# Dynamic path resolution: Kaggle (competition input path) vs Local
if os.path.exists('/kaggle/input/competitions/playground-series-s6e6'):
    DATA_DIR = '/kaggle/input/competitions/playground-series-s6e6'
else:
    DATA_DIR = '../data'

train_path = os.path.join(DATA_DIR, 'train.csv')
test_path = os.path.join(DATA_DIR, 'test.csv')
submission_path = os.path.join(DATA_DIR, 'sample_submission.csv')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print(f"Data Directory in use: {DATA_DIR}\n")
print("--- Dataset Shapes ---")
print(f"Train shape:             {train.shape}")
print(f"Test shape:              {test.shape}")
print(f"Sample submission shape: {submission.shape}")

## 2. Dataset Feature Profiling

Previewing the raw data schema. The target variable is the categorical feature `class` (`GALAXY`, `STAR`, `QSO`). Features consist of coordinates (`alpha`, `delta`), photometric bands (`u`, `g`, `r`, `i`, `z`), and catalog categoricals (`spectral_type`, `galaxy_population`).

In [ ]:
print("\n--- Train Dataset Head ---")
display(train.head())

print("\n--- Test Dataset Head ---")
display(test.head())

print("\n--- Sample Submission Dataset Head ---")
display(submission.head())

## 3. Target Class Imbalance & Metric Mechanics

The target variable `class` exhibits significant distribution skew:
- `GALAXY`: ~65.4% (Majority class)
- `QSO` (Quasar): ~20.3%
- `STAR`: ~14.3% (Minority class)

### Balanced Accuracy Metric Formulation
Because of this imbalance, the competition uses **Balanced Accuracy** as the primary evaluation metric. Mathematically:
$$\text{Balanced Accuracy} = \frac{1}{3} \left( \text{Recall}_{\text{Galaxy}} + \text{Recall}_{\text{QSO}} + \text{Recall}_{\text{Star}} \right)$$
where each class recall is defined as:
$$\text{Recall}_c = \frac{\text{True Positives}_c}{\text{True Positives}_c + \text{False Negatives}_c}$$

### Modeling Strategy
Standard cross-entropy loss optimization will bias models toward the majority `GALAXY` class. To counter this, we must align our validation metrics with the unweighted recall average. A misclassified `STAR` is mathematically **4.5x** more expensive than a misclassified `GALAXY` (\frac{65.4\%}{14.3\%} \approx 4.57$). We will implement sample-weight and class-weight balancing configurations inside our model training scripts.

In [ ]:
counts = train['class'].value_counts()
pcts = train['class'].value_counts(normalize=True) * 100
dist_df = pd.DataFrame({'Count': counts, 'Percentage': pcts})
print(dist_df)

plt.figure(figsize=(6, 4))
sns.countplot(data=train, x='class', hue='class', palette='viridis', legend=False)
plt.title('Target Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Cosmological Context of Redshift ($z$)

Redshift ($z$) measures the stretching of light wavelengths toward redder frequencies due to the expansion of the universe (Hubble's Law). The astronomical profiles of our classes dictate their redshift distributions:

$$\text{Redshift } z = \frac{\lambda_{\text{observed}} - \lambda_{\text{emit}}}{\lambda_{\text{emit}}}$$

### Astrophysical Profile
1. **Stars ($z \approx 0$):** Stars are gravitationally bound within the Milky Way galaxy. They do not expand with the cosmic web, resulting in a redshift distribution tightly centered around 0 (median: $0.00007$).
2. **Galaxies ($0 < z \le 1.0$):** Galaxies reside outside our local group and recede from us, exhibiting low-to-mid range redshift values corresponding to light travel distance (median: $0.077$).
3. **Quasars / QSOs ($1.5 \le z \le 7.0$):** Quasars are highly energetic active galactic nuclei (AGN) powered by supermassive black holes. Visible only at cosmological distances, they exhibit high redshift distributions (median: $1.72$, stretching up to $7+$).

### Decision Boundaries & Overlaps
- **Galaxy/Star Overlap ($z \le 0$):** Local group galaxies (such as Andromeda) can exhibit negative redshift due to blueshift. Separating local galaxies from stars requires relying on photometric magnitudes.
- **Galaxy/QSO Overlap ($z \ge 1.0$):** Starburst galaxies at extreme distances show redshifts that overlap with the quasar region. We must utilize spectral indicators to partition these targets.

In [ ]:
print("--- Redshift Summary by Class ---")
print(train.groupby('class')['redshift'].describe())

plt.figure(figsize=(10, 5))
sns.histplot(data=train, x='redshift', hue='class', kde=True, bins=100, palette='viridis', multiple='stack')
plt.title('Redshift Distribution by Class')
plt.xlabel('Redshift')
plt.ylabel('Count')
plt.yscale('log') # Using log scale due to broad range and peak density differences
plt.tight_layout()
plt.show()

## 5. Photometric Color Indices as Temperature Proxies

Magnitudes represent the brightness of light captured through filters ($u$, $g$, $r$, $i$, $z$). Direct magnitudes depend heavily on object distance and absolute luminosity. To extract intrinsic surface properties, astronomers subtract adjacent filter magnitudes to form **Color Indices** (e.g., $u-g$, $g-r$, $r-i$, $i-z$).

### Physical Interpretation
- **Blackbody Stellar Curves:** Stars radiate thermal energy close to a blackbody curve. Cool stars (Class M) peak in redder bands (large $u-g$), while hot stars (Class O/B) emit predominantly in UV/blue bands (low/negative $u-g$).
- **Non-Thermal Quasar Power-Laws:** Quasars display a power-law spectrum driven by gravitational accretion disks, overlaid with strong Lyman-alpha emission lines. This places them in unique sectors of multi-dimensional color space (e.g. $u-g$ vs $g-r$ color-color plot).

In [ ]:
# Engineering color index differences
train['u_g'] = train['u'] - train['g']
train['g_r'] = train['g'] - train['r']
train['r_i'] = train['r'] - train['i']
train['i_z'] = train['i'] - train['z']

# Sample data for plotting to keep memory/runtime small
sample = train.sample(20000, random_state=42)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=sample, x='g_r', y='u_g', hue='class', palette='viridis', alpha=0.4, s=10)
plt.title('Color-Color Diagram: u-g vs g-r (Sample of 20k)')
plt.xlabel('g - r')
plt.ylabel('u - g')
plt.tight_layout()
plt.show()

## 6. Spherical Geometry & Coordinate Discontinuity Resolution

Positional features are recorded in Right Ascension (\alpha or `alpha`) and Declination (\delta or `delta`). Right Ascension is a circular angle ranging from $0^\circ$ to $360^\circ$.

### The Boundary Wrapping Problem
Decision tree algorithms make orthogonal axis splits. When dividing on a circular feature like `alpha`, a boundary split at $0^\circ / 360^\circ$ splits adjacent spatial coordinates (e.g., $359.9^\circ$ and $0.1^\circ$ are physically close but placed at opposite ends of the feature scale). This forces trees to construct unnecessary splits to merge spatial regions.

### 3D Unit Sphere Projection
To resolve this discontinuity, we project the angles into Cartesian coordinates on a 3D unit sphere:
$$x = \cos(\delta) \cos(\alpha)$$
$$y = \cos(\delta) \sin(\alpha)$$
$$z = \sin(\delta)$$
*(Note: Angles must be converted to radians).* This maps angular spatial coordinates into a continuous coordinate space, permitting decision trees to split on local spatial proximity.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=sample, x='alpha', y='delta', hue='class', palette='viridis', alpha=0.3, s=8)
plt.title('Celestial Positions (Sample of 20k)')
plt.xlabel('Alpha (Right Ascension)')
plt.ylabel('Delta (Declination)')
plt.tight_layout()
plt.show()

## 7. Categorical Spectral Types & Galaxy Populations

The dataset includes categorical features mapping physical sub-types:
1. **Spectral Type (`spectral_type`):** Identifies temperature classes (`O`, `B`, `A`, `F`, `G`, `K`, `M`). Hot classes (`O/B`) indicate young massive stars or energetic AGN (Quasars). Passive classes (`M`) correlate with cool stars or red sequence galaxies.
2. **Galaxy Population (`galaxy_population`):** Splits galaxies into the passively evolving **Red Sequence** (mostly older elliptical galaxies) and star-forming **Blue Cloud** populations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=train, x='spectral_type', hue='class', palette='viridis', ax=axes[0])
axes[0].set_title('Spectral Type by Target Class')
axes[0].set_xlabel('Spectral Type')

sns.countplot(data=train, x='galaxy_population', hue='class', palette='viridis', ax=axes[1])
axes[1].set_title('Galaxy Population by Target Class')
axes[1].set_xlabel('Galaxy Population')

plt.tight_layout()
plt.show()